# Cybersecurity Mean-Field Control
## Simplex MF-REINFORCE Only

This notebook is a fixed-simplex-only copy of the cybersecurity numerical experiment. It uses exact population recursion for validation throughout, with either exact or particle-estimated population flow during training.

### Experiments

1. **Nominal-budget exact flow.** Simplex uses `(B, n_aux) = (200, 10)`.
2. **Full-budget exact flow.** Simplex uses `(B, n_aux) = (3675, 525)`, the paper-scale simplex budget for `T_train=3`.
3. **Full-budget estimated flow.** The same simplex budget is used with a particle flow estimate using `flow_particles=200`.

The saved notebook is output-free and ready to run from top to bottom.


## Imports


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import copy
import math
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm import tqdm

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = (ROOT / "..").resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from mfc.algorithms import SimplexPerturbedMFREINFORCE
from mfc.environments import CybersecurityConfig, CybersecurityMFC, CybersecurityPolicy


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
torch.set_default_dtype(DTYPE)


## Runtime And General Helpers


In [3]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)


set_seed(0)
print(f"device: {device}")


def format_runtime(seconds: Optional[float]) -> str:
    if seconds is None:
        return "not set"
    seconds = float(seconds)
    hours, remainder = divmod(int(seconds), 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours:d}h {minutes:02d}m {secs:02d}s"
    if minutes:
        return f"{minutes:d}m {secs:02d}s"
    return f"{seconds:.1f}s"


def std_ddof(n: int) -> int:
    return 1 if n > 1 else 0


def safe_name(name: object) -> str:
    text = str(name)
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in text).strip("_") or "cybersecurity"


def tensor_float(value, default: float = float("nan")) -> float:
    if value is None:
        return default
    if isinstance(value, torch.Tensor):
        if value.numel() != 1:
            return default
        return float(value.detach().cpu().item())
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def aligned_validation_history(runs):
    min_len = min(len(run["history"]["validation_value"]) for run in runs)
    episodes = np.asarray(runs[0]["history"]["episode"][:min_len], dtype=float)
    values = np.asarray([run["history"]["validation_value"][:min_len] for run in runs], dtype=float)
    return episodes, values


def aligned_history_metric(runs, key: str):
    lengths = [len(run["history"].get(key, [])) for run in runs]
    if not lengths or min(lengths) == 0:
        return np.asarray([]), np.asarray([])
    min_len = min(lengths)
    episodes = np.asarray(runs[0]["history"]["episode"][:min_len], dtype=float)
    values = np.asarray([run["history"][key][:min_len] for run in runs], dtype=float)
    return episodes, values


device: cuda


## Policy, Run Plan, And Flow Helpers


In [4]:
def fixed_validation_law(config: CybersecurityConfig) -> torch.Tensor:
    return torch.full(
        (config.n_states,),
        1.0 / config.n_states,
        dtype=config.dtype,
        device=config.device,
    )


def sample_cybersecurity_initial_laws(config: CybersecurityConfig, count: int) -> torch.Tensor:
    concentration = torch.ones(config.n_states, dtype=config.dtype, device=config.device)
    return torch.distributions.Dirichlet(concentration).sample((count,)).detach().cpu()


def clone_state_dict(policy: CybersecurityPolicy) -> Dict[str, torch.Tensor]:
    return {key: value.detach().cpu().clone() for key, value in policy.state_dict().items()}


def prepare_paired_run_plans(
    config: CybersecurityConfig,
    seed_base: int = 51_000,
    training_runs: Optional[int] = None,
) -> List[Dict[str, object]]:
    n_runs = config.training_runs if training_runs is None else int(training_runs)
    plans: List[Dict[str, object]] = []
    for run_idx in range(n_runs):
        seed = seed_base + run_idx
        set_seed(seed)
        policy = CybersecurityPolicy(config)
        plans.append(
            {
                "run_idx": run_idx,
                "seed": seed,
                "initial_control": {"state_dict": clone_state_dict(policy)},
                "initial_laws": sample_cybersecurity_initial_laws(config, config.n_train),
            }
        )
    return plans


def load_policy(config: CybersecurityConfig, payload: Dict[str, object], trainable: bool = True) -> CybersecurityPolicy:
    policy = CybersecurityPolicy(config)
    state = {
        key: value.to(dtype=config.dtype, device=config.device).detach().clone()
        for key, value in payload["state_dict"].items()
    }
    policy.load_state_dict(state)
    policy.train(trainable)
    for parameter in policy.parameters():
        parameter.requires_grad_(trainable)
    return policy


def payload_from_record(record: Dict[str, object]) -> Dict[str, object]:
    return {"state_dict": {key: value.detach().cpu().clone() for key, value in record["policy_state_dict"].items()}}


def parameter_vector(policy: CybersecurityPolicy) -> torch.Tensor:
    return torch.nn.utils.parameters_to_vector(policy.parameters()).detach()


def assign_flat_ascent_gradient(policy: CybersecurityPolicy, grad_hat: torch.Tensor) -> None:
    grad_flat = grad_hat.detach().reshape(-1)
    offset = 0
    for parameter in policy.parameters():
        count = parameter.numel()
        parameter.grad = -grad_flat[offset : offset + count].reshape_as(parameter).clone()
        offset += count
    if offset != grad_flat.numel():
        raise ValueError(f"Gradient length {grad_flat.numel()} does not match policy parameter length {offset}.")


@torch.no_grad()
def exact_population_flow_batch(
    env: CybersecurityMFC,
    policy: CybersecurityPolicy,
    mu0_batch: torch.Tensor,
    horizon: int,
) -> torch.Tensor:
    flow = [mu0_batch]
    for t in range(horizon):
        kernel = env.averaged_kernel(policy, t, flow[-1])
        flow.append(torch.einsum("bi,bij->bj", flow[-1], kernel))
    return torch.stack(flow, dim=1)


def training_population_flow(
    env: CybersecurityMFC,
    algorithm,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    horizon: int,
    flow_mode: str,
    flow_particles: int,
) -> torch.Tensor:
    if flow_mode == "exact":
        with torch.no_grad():
            return env.exact_population_flow(policy, mu0, horizon).detach()
    if flow_mode == "particle":
        return algorithm.estimate_population_flow(policy, mu0, flow_particles, horizon=horizon).detach()
    raise ValueError(f"Unknown flow_mode={flow_mode!r}.")


## Metrics, Costs, And Perturbation Calibration


In [5]:
def reference_metrics(
    env: CybersecurityMFC,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    horizon: int,
) -> Dict[str, object]:
    was_training = policy.training
    policy.eval()
    try:
        with torch.no_grad():
            flow = env.exact_population_flow(policy, mu0, horizon).detach()
            value = env.exact_value(policy, mu0, horizon).detach()
            infected = flow[:, env.config.DI] + flow[:, env.config.UI]
            defended = flow[:, env.config.DI] + flow[:, env.config.DS]
            update_probs = []
            for t in range(horizon):
                pi = env.action_probabilities(policy, t, flow[t])
                update_probs.append((flow[t] * pi[:, env.config.UPDATE]).sum())
            update_probs_t = torch.stack(update_probs) if update_probs else torch.zeros(0, dtype=flow.dtype, device=flow.device)
    finally:
        policy.train(was_training)

    final_distribution = flow[-1].detach().cpu()
    state_names = env.config.cyber_state_names
    metrics: Dict[str, object] = {
        "value": float(value.item()),
        "flow": flow.detach().cpu(),
        "infected": infected.detach().cpu(),
        "defended": defended.detach().cpu(),
        "update_probability": update_probs_t.detach().cpu(),
        "final_distribution": final_distribution,
        "terminal_infected": float(infected[-1].item()),
        "terminal_defended": float(defended[-1].item()),
        "mean_infected": float(infected.mean().item()),
        "mean_defended": float(defended.mean().item()),
        "mean_update_probability": float(update_probs_t.mean().item()) if update_probs_t.numel() else float("nan"),
    }
    for idx, state_name in enumerate(state_names):
        metrics[f"terminal_{state_name}"] = float(final_distribution[idx].item())
    return metrics


def record_numeric_history(history: Dict[str, List[float]], metrics: Dict[str, object]) -> None:
    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)):
            history.setdefault(key, []).append(float(value))


def simulator_transitions_per_update(
    algorithm_name: str,
    horizon: int,
    B: int,
    n_aux_or_inner: int,
    flow_mode: str = "exact",
    flow_particles: int = 0,
) -> int:
    normalized = algorithm_name.lower().replace("-", "").replace("_", "").replace(" ", "")
    if normalized not in {"simplex", "fixedsimplex"}:
        raise ValueError(f"This notebook only supports fixed simplex MF-REINFORCE, got {algorithm_name!r}.")
    core = (int(B) + int(n_aux_or_inner)) * int(horizon)
    flow_cost = 0 if flow_mode == "exact" else int(flow_particles) * int(horizon)
    return int(math.ceil(core + flow_cost))


def simulator_cost_table(
    horizon: int,
    flow_mode: str,
    budgets: Dict[str, Dict[str, int]],
    flow_particles: int = 0,
) -> pd.DataFrame:
    rows = []
    for algorithm_name, budget in budgets.items():
        cost = simulator_transitions_per_update(
            algorithm_name,
            horizon,
            budget["B"],
            budget["n"],
            flow_mode=flow_mode,
            flow_particles=flow_particles,
        )
        rows.append(
            {
                "algorithm": algorithm_name,
                "B": budget["B"],
                "n": budget["n"],
                "horizon": horizon,
                "flow_mode": flow_mode,
                "flow_particles": flow_particles if flow_mode == "particle" else 0,
                "simulator_transitions_per_update": cost,
            }
        )
    return pd.DataFrame(rows).set_index("algorithm")


def sample_simplex_q(config: CybersecurityConfig, n: int) -> torch.Tensor:
    u = config.q_sigma * torch.randn(n, config.n_states - 1, dtype=config.dtype, device=config.device)
    raw_scores = torch.cat([u, torch.zeros(n, 1, dtype=config.dtype, device=config.device)], dim=-1)
    q = torch.softmax(raw_scores, dim=-1).clamp_min(config.q_clip)
    return q / q.sum(dim=-1, keepdim=True)


def collect_reference_laws(
    config: CybersecurityConfig,
    payload: Dict[str, object],
    count: int,
    horizon: int,
    chunk_size: int = 512,
) -> torch.Tensor:
    env = CybersecurityMFC(config)
    policy = load_policy(config, payload, trainable=False)
    initial_laws = sample_cybersecurity_initial_laws(config, count).to(dtype=config.dtype, device=config.device)
    chunks = []
    for start in range(0, count, chunk_size):
        mu0_batch = initial_laws[start : start + chunk_size]
        flow = exact_population_flow_batch(env, policy, mu0_batch, horizon)
        chunks.append(flow.reshape(-1, config.n_states).detach().cpu())
    return torch.cat(chunks, dim=0)


def calibrate_simplex_radii(
    config: CybersecurityConfig,
    reference_laws: torch.Tensor,
    simplex_values: Sequence[float],
    num_samples: int = 20_000,
) -> pd.DataFrame:
    reference_laws = reference_laws.to(dtype=config.dtype, device=config.device)
    index = torch.randint(reference_laws.shape[0], (num_samples,), device=config.device)
    mu = reference_laws[index].clamp_min(config.q_clip)
    mu = mu / mu.sum(dim=-1, keepdim=True)
    q = sample_simplex_q(config, num_samples)
    rows = []

    for value in simplex_values:
        perturbed = (1.0 - float(value)) * mu + float(value) * q
        radii = 0.5 * (mu - perturbed).abs().sum(dim=-1).detach().cpu().numpy()
        rows.append(
            {
                "algorithm": "Simplex",
                "parameter": float(value),
                "tv_mean": radii.mean(),
                "tv_median": np.median(radii),
                "tv_std": radii.std(ddof=0),
                "tv_p10": np.quantile(radii, 0.10),
                "tv_p90": np.quantile(radii, 0.90),
            }
        )
    return pd.DataFrame(rows)


## Training Runners


In [6]:
ALGORITHM_NAME = "Simplex"


def train_simplex_cybersecurity_method(
    config: CybersecurityConfig,
    perturbation_values: Sequence[float],
    budget: Dict[str, int],
    run_plans: List[Dict[str, object]],
    flow_mode: str,
    flow_particles: int,
    train_horizon: int,
    validation_horizon: int,
    label: str,
    show_progress: bool = True,
    early_stopping_patience: Optional[int] = None,
    early_stopping_min_delta: float = 0.0,
    max_runtime_seconds: Optional[float] = None,
) -> Dict[float, List[Dict[str, object]]]:
    B = int(budget["B"])
    n_aux = int(budget["n"])
    fixed_mu0 = fixed_validation_law(config)
    results: Dict[float, List[Dict[str, object]]] = {}

    for perturbation in perturbation_values:
        parameter_key = float(perturbation)
        results[parameter_key] = []
        for run_idx, plan in enumerate(run_plans):
            set_seed(int(plan["seed"]))
            env = CybersecurityMFC(config)
            policy = load_policy(config, plan["initial_control"], trainable=True)
            optimizer = torch.optim.Adam(policy.parameters(), lr=config.lr)
            algorithm = SimplexPerturbedMFREINFORCE(env)
            history: Dict[str, List[float]] = {
                "episode": [],
                "validation_value": [],
                "train_return_mean": [],
                "grad_norm": [],
                "lambda": [],
                "eta": [],
                "cumulative_simulator_transitions": [],
                "elapsed_seconds": [],
            }
            cumulative_transitions = 0
            run_start = time.perf_counter()
            stop_reason = "completed"
            episodes_completed = 0
            best_validation = -float("inf")
            best_episode = None
            best_state_dict = None
            checks_since_best = 0

            iterator = range(config.n_train)
            if show_progress:
                iterator = tqdm(iterator, desc=f"{label} {ALGORITHM_NAME} lambda={parameter_key:g} run={run_idx}")

            for episode in iterator:
                if max_runtime_seconds is not None and time.perf_counter() - run_start >= max_runtime_seconds:
                    stop_reason = f"max_runtime_{format_runtime(max_runtime_seconds)}"
                    break

                mu0 = plan["initial_laws"][episode].to(dtype=config.dtype, device=config.device)
                mu_flow = training_population_flow(
                    env,
                    algorithm,
                    policy,
                    mu0,
                    train_horizon,
                    flow_mode,
                    flow_particles,
                )
                grad_hat, diag = algorithm.complete_gradient_estimate(
                    policy,
                    mu_flow,
                    parameter_key,
                    B,
                    n_aux,
                    eta=parameter_key,
                    baseline="batch_mean",
                )

                transition_value = tensor_float(diag.get("simulator_transitions"), default=float("nan"))
                if math.isnan(transition_value):
                    transitions = simulator_transitions_per_update(
                        ALGORITHM_NAME,
                        train_horizon,
                        B,
                        n_aux,
                        flow_mode,
                        flow_particles,
                    )
                else:
                    transitions = int(transition_value)
                cumulative_transitions += transitions

                optimizer.zero_grad(set_to_none=True)
                assign_flat_ascent_gradient(policy, grad_hat)
                optimizer.step()
                episodes_completed = episode + 1

                if episode % config.validate_every == 0 or episode == config.n_train - 1:
                    metrics = reference_metrics(env, policy, fixed_mu0, validation_horizon)
                    validation_value = metrics["value"]
                    history["episode"].append(float(episode))
                    history["validation_value"].append(validation_value)
                    history["train_return_mean"].append(tensor_float(diag.get("mean_return")))
                    history["grad_norm"].append(tensor_float(diag.get("grad_norm")))
                    history["lambda"].append(parameter_key)
                    history["eta"].append(parameter_key)
                    history["cumulative_simulator_transitions"].append(float(cumulative_transitions))
                    history["elapsed_seconds"].append(time.perf_counter() - run_start)
                    record_numeric_history(history, metrics)

                    if validation_value > best_validation + early_stopping_min_delta:
                        best_validation = validation_value
                        best_episode = episode
                        best_state_dict = clone_state_dict(policy)
                        checks_since_best = 0
                    else:
                        checks_since_best += 1

                    if show_progress:
                        iterator.set_postfix(value=f"{validation_value:.4g}", grad=f"{history['grad_norm'][-1]:.3g}")

                    if early_stopping_patience is not None and checks_since_best >= early_stopping_patience:
                        stop_reason = "early_stopping"
                        break

            final_metrics = reference_metrics(env, policy, fixed_mu0, validation_horizon)
            record = {
                "algorithm": ALGORITHM_NAME,
                "method": "simplex",
                "flow_mode": flow_mode,
                "flow_particles": flow_particles if flow_mode == "particle" else 0,
                "parameter": parameter_key,
                "run_idx": run_idx,
                "seed": int(plan["seed"]),
                "policy_state_dict": clone_state_dict(policy),
                "history": history,
                "final_value": final_metrics["value"],
                "reference_metrics": final_metrics,
                "runtime_seconds": time.perf_counter() - run_start,
                "episodes_completed": episodes_completed,
                "main_trajectories": B,
                "auxiliary_trajectories": n_aux,
                "simulator_transitions_per_update": simulator_transitions_per_update(
                    ALGORITHM_NAME,
                    train_horizon,
                    B,
                    n_aux,
                    flow_mode,
                    flow_particles,
                ),
                "total_simulator_transitions": cumulative_transitions,
                "stop_reason": stop_reason,
                "best_validation_value": best_validation,
                "best_episode": best_episode,
                "best_policy_state_dict": best_state_dict,
            }
            results[parameter_key].append(record)
            print(
                f"{label} {ALGORITHM_NAME} lambda={parameter_key:g} run={run_idx} "
                f"value={record['final_value']:.6g} transitions={record['total_simulator_transitions']} "
                f"runtime={format_runtime(record['runtime_seconds'])}"
            )
    return results


def run_cybersecurity_scenario(
    name: str,
    config: CybersecurityConfig,
    simplex_lambdas: Sequence[float],
    budgets: Dict[str, Dict[str, int]],
    run_plans: List[Dict[str, object]],
    flow_mode: str,
    flow_particles: int = 0,
    train_horizon: Optional[int] = None,
    validation_horizon: Optional[int] = None,
    show_progress: bool = True,
    early_stopping_patience: Optional[int] = None,
    max_runtime_seconds: Optional[float] = None,
) -> Dict[str, object]:
    train_horizon = config.T_train if train_horizon is None else int(train_horizon)
    validation_horizon = config.T_val if validation_horizon is None else int(validation_horizon)
    costs = simulator_cost_table(train_horizon, flow_mode, budgets, flow_particles=flow_particles)
    display(costs.round(4))
    simplex_results = train_simplex_cybersecurity_method(
        config,
        simplex_lambdas,
        budgets["Simplex"],
        run_plans,
        flow_mode,
        flow_particles,
        train_horizon,
        validation_horizon,
        label=name,
        show_progress=show_progress,
        early_stopping_patience=early_stopping_patience,
        max_runtime_seconds=max_runtime_seconds,
    )
    return {
        "name": name,
        "flow_mode": flow_mode,
        "flow_particles": flow_particles if flow_mode == "particle" else 0,
        "budgets": copy.deepcopy(budgets),
        "costs": costs,
        "train_horizon": train_horizon,
        "validation_horizon": validation_horizon,
        "results": {"Simplex": simplex_results},
    }


## Reporting And Plotting


In [7]:
def metric_from_run(run: Dict[str, object], key: str) -> float:
    if key in run.get("reference_metrics", {}):
        return float(run["reference_metrics"][key])
    return float(run[key])


def summarize_metric_rows(result_groups, metric_keys: Sequence[str]) -> pd.DataFrame:
    rows = []
    for algorithm_name, result_group in result_groups.items():
        for perturbation, runs in result_group.items():
            row = {"algorithm": algorithm_name, "parameter": float(perturbation), "runs": len(runs)}
            ddof = std_ddof(len(runs))
            for key in metric_keys:
                values = np.asarray([metric_from_run(run, key) for run in runs], dtype=float)
                row[f"{key}_mean"] = float(values.mean())
                row[f"{key}_std"] = float(values.std(ddof=ddof))
            for key in [
                "runtime_seconds",
                "episodes_completed",
                "main_trajectories",
                "auxiliary_trajectories",
                "simulator_transitions_per_update",
                "total_simulator_transitions",
                "best_validation_value",
            ]:
                if key in runs[0]:
                    row[f"{key}_mean"] = float(np.mean([run[key] for run in runs]))
            rows.append(row)
    return pd.DataFrame(rows).set_index(["algorithm", "parameter"]).sort_index()


def radius_summary(result_groups, calibration: pd.DataFrame, metric_key: str = "value") -> pd.DataFrame:
    rows = []
    for algorithm_name, result_group in result_groups.items():
        for perturbation, runs in result_group.items():
            radius_row = calibration[
                (calibration["algorithm"] == algorithm_name)
                & np.isclose(calibration["parameter"], float(perturbation))
            ]
            if radius_row.empty:
                continue
            values = np.asarray([metric_from_run(run, metric_key) for run in runs], dtype=float)
            rows.append(
                {
                    "algorithm": algorithm_name,
                    "parameter": float(perturbation),
                    "radius": float(radius_row.iloc[0]["tv_mean"]),
                    f"{metric_key}_mean": values.mean(),
                    f"{metric_key}_std": values.std(ddof=std_ddof(len(values))),
                }
            )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index(["algorithm", "parameter"]).sort_index()


LINE_STYLES = {"Simplex": "-"}


def plot_validation_groups(result_groups, title: str):
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    keys = [(algorithm_name, parameter) for algorithm_name, group in result_groups.items() for parameter in group]
    cmap = plt.get_cmap("tab20", max(len(keys), 1))
    colors = {key: cmap(idx) for idx, key in enumerate(keys)}
    for algorithm_name, result_group in result_groups.items():
        for perturbation, runs in result_group.items():
            episodes, values = aligned_validation_history(runs)
            mean = values.mean(axis=0)
            std = values.std(axis=0, ddof=std_ddof(len(runs)))
            color = colors[(algorithm_name, perturbation)]
            ax.plot(
                episodes,
                mean,
                linestyle=LINE_STYLES.get(algorithm_name, "-"),
                color=color,
                label=f"{algorithm_name}, lambda={perturbation:g}",
            )
            ax.fill_between(episodes, mean - std, mean + std, color=color, alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("Training iteration")
    ax.set_ylabel("Exact validation value (mean +/- std. dev.)")
    ax.grid(alpha=0.25)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    plt.show()


def best_parameter_for_group(result_group: Dict[float, List[Dict[str, object]]]) -> float:
    rows = []
    for parameter, runs in result_group.items():
        values = np.asarray([run["final_value"] for run in runs], dtype=float)
        rows.append(
            {
                "parameter": float(parameter),
                "value_mean": values.mean(),
                "value_std": values.std(ddof=std_ddof(len(values))),
            }
        )
    return float(pd.DataFrame(rows).sort_values(["value_mean", "value_std", "parameter"], ascending=[False, True, True]).iloc[0]["parameter"])


def plot_best_population_flows(result_groups, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for algorithm_name, result_group in result_groups.items():
        parameter = best_parameter_for_group(result_group)
        runs = result_group[parameter]
        flows = np.asarray([run["reference_metrics"]["flow"].numpy() for run in runs], dtype=float)
        mean_flow = flows.mean(axis=0)
        infected = mean_flow[:, 0] + mean_flow[:, 2]
        defended = mean_flow[:, 0] + mean_flow[:, 1]
        times = np.arange(mean_flow.shape[0])
        label = f"{algorithm_name}, lambda={parameter:g}"
        axes[0].plot(times, infected, label=label)
        axes[1].plot(times, defended, label=label)
    axes[0].set_title("Infected fraction")
    axes[1].set_title("Defended fraction")
    for ax in axes:
        ax.set_xlabel("Validation time")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def scenario_summary_table(scenario: Dict[str, object], experiment: str, horizon: Optional[int] = None) -> pd.DataFrame:
    table = summarize_metric_rows(
        scenario["results"],
        [
            "value",
            "terminal_infected",
            "terminal_defended",
            "mean_infected",
            "mean_defended",
            "mean_update_probability",
        ],
    ).reset_index()
    table.insert(0, "experiment", experiment)
    table.insert(1, "flow_mode", scenario["flow_mode"])
    table.insert(2, "T_train", int(scenario["train_horizon"] if horizon is None else horizon))
    table.insert(3, "flow_particles", int(scenario["flow_particles"]))
    if "costs" in scenario and "simulator_transitions_per_update" in scenario["costs"].columns:
        cost_lookup = scenario["costs"]["simulator_transitions_per_update"].to_dict()
        table["scenario_cost_per_update"] = table["algorithm"].map(cost_lookup)
    leading_columns = ["experiment", "flow_mode", "T_train", "flow_particles", "algorithm", "parameter"]
    return table[leading_columns + [column for column in table.columns if column not in leading_columns]]


def show_cybersecurity_results(scenario: Dict[str, object], calibration: pd.DataFrame):
    result_groups = scenario["results"]
    print(scenario["name"])
    display(scenario["costs"].round(4))
    display(calibration.round(4))
    display(radius_summary(result_groups, calibration, metric_key="value").round(6))
    plot_validation_groups(result_groups, f"{scenario['name']}: validation value")
    display(
        summarize_metric_rows(
            result_groups,
            [
                "value",
                "terminal_DI",
                "terminal_DS",
                "terminal_UI",
                "terminal_US",
                "terminal_infected",
                "terminal_defended",
                "mean_update_probability",
            ],
        ).round(6)
    )
    plot_best_population_flows(result_groups, f"{scenario['name']}: best population flows")


## Exact-Gradient Diagnostics


In [8]:
def exact_gradient(
    env: CybersecurityMFC,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    horizon: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    for parameter in policy.parameters():
        parameter.requires_grad_(True)
    value = env.exact_value(policy, mu0, horizon)
    grads = torch.autograd.grad(value, tuple(policy.parameters()), allow_unused=False)
    flat_grad = torch.cat([grad.detach().reshape(-1) for grad in grads])
    return value.detach(), flat_grad


def cosine_similarity_flat(x: torch.Tensor, y: torch.Tensor) -> float:
    denom = torch.linalg.norm(x) * torch.linalg.norm(y)
    if float(denom.item()) == 0.0:
        return float("nan")
    return float((x.flatten() @ y.flatten() / denom).item())


def simplex_gradient_diagnostic_summary(
    config: CybersecurityConfig,
    payload: Dict[str, object],
    perturbation: float,
    budget: Dict[str, int],
    mu0: torch.Tensor,
    horizon: int,
    flow_mode: str,
    flow_particles: int,
    repetitions: int,
    label: str,
    show_progress: bool = True,
) -> Dict[str, object]:
    env = CybersecurityMFC(config)
    policy = load_policy(config, payload, trainable=True)
    mu0 = mu0.to(dtype=config.dtype, device=config.device)
    B = int(budget["B"])
    n_aux = int(budget["n"])
    _, oracle_grad = exact_gradient(env, policy, mu0, horizon)
    per_estimate_transitions = simulator_transitions_per_update(
        ALGORITHM_NAME,
        horizon,
        B,
        n_aux,
        flow_mode=flow_mode,
        flow_particles=flow_particles,
    )

    samples = []
    iterator = range(repetitions)
    if show_progress:
        iterator = tqdm(iterator, desc=f"{label} {ALGORITHM_NAME} lambda={perturbation:g}", leave=False)
    algorithm = SimplexPerturbedMFREINFORCE(env)

    for _ in iterator:
        mu_flow = training_population_flow(env, algorithm, policy, mu0, horizon, flow_mode, flow_particles)
        grad_hat, _ = algorithm.complete_gradient_estimate(
            policy,
            mu_flow,
            float(perturbation),
            B,
            n_aux,
            eta=float(perturbation),
            baseline="batch_mean",
        )
        samples.append(grad_hat.detach().reshape(-1))

    samples_t = torch.stack(samples)
    mean_grad = samples_t.mean(dim=0)
    bias = mean_grad - oracle_grad
    covariance_trace = float(samples_t.var(dim=0, unbiased=repetitions > 1).sum().item()) if repetitions > 1 else 0.0
    mse = ((samples_t - oracle_grad.unsqueeze(0)).square().sum(dim=1)).mean()
    return {
        "label": label,
        "algorithm": ALGORITHM_NAME,
        "flow_mode": flow_mode,
        "flow_particles": flow_particles if flow_mode == "particle" else 0,
        "parameter": float(perturbation),
        "repetitions": repetitions,
        "B": B,
        "n_aux_or_inner": n_aux,
        "oracle_grad_norm": float(torch.linalg.norm(oracle_grad).item()),
        "mean_grad_norm": float(torch.linalg.norm(mean_grad).item()),
        "bias_norm": float(torch.linalg.norm(bias).item()),
        "covariance_trace": covariance_trace,
        "mse": float(mse.item()),
        "cosine_to_oracle": cosine_similarity_flat(mean_grad, oracle_grad),
        "simulator_transitions_per_estimate": int(per_estimate_transitions),
        "simulator_transitions": int(per_estimate_transitions * repetitions),
    }


def run_cybersecurity_diagnostics(
    scenario: Dict[str, object],
    config: CybersecurityConfig,
    simplex_lambdas: Sequence[float],
    run_plans: List[Dict[str, object]],
    repetitions: int = 8,
    show_progress: bool = True,
) -> pd.DataFrame:
    results = scenario["results"]
    best_simplex = best_parameter_for_group(results["Simplex"])
    controls = [
        ("initial", run_plans[0]["initial_control"]),
        (f"simplex_final_lambda_{best_simplex:g}", payload_from_record(results["Simplex"][best_simplex][0])),
    ]
    rows = []
    mu0 = fixed_validation_law(config)
    for label, payload in controls:
        for perturbation in simplex_lambdas:
            rows.append(
                simplex_gradient_diagnostic_summary(
                    config,
                    payload,
                    perturbation,
                    scenario["budgets"]["Simplex"],
                    mu0,
                    scenario["train_horizon"],
                    scenario["flow_mode"],
                    scenario["flow_particles"],
                    repetitions,
                    label,
                    show_progress=show_progress,
                )
            )
    return pd.DataFrame(rows)


## Common Experiment Setup


In [9]:
config = CybersecurityConfig(device=device, dtype=DTYPE)
train_horizon = config.T_train
validation_horizon = config.T_val

simplex_lambdas = [0.1, 0.2, 0.4]

nominal_simplex_budgets = {"Simplex": {"B": 200, "n": 10}}
full_simplex_budgets = {"Simplex": {"B": 3675, "n": 525}}
particle_flow_particles = 200

early_stopping_patience = None
max_runtime_seconds = None
diagnostic_repetitions = 8
perturbation_calibration_samples = 20_000
reference_law_count = 20_000

cybersecurity_run_plans = prepare_paired_run_plans(config, seed_base=51_000)
cybersecurity_reference_laws = collect_reference_laws(
    config,
    cybersecurity_run_plans[0]["initial_control"],
    reference_law_count,
    train_horizon,
)
cybersecurity_calibration = calibrate_simplex_radii(
    config,
    cybersecurity_reference_laws,
    simplex_lambdas,
    num_samples=perturbation_calibration_samples,
)

nominal_exact_costs = simulator_cost_table(train_horizon, "exact", nominal_simplex_budgets)
full_exact_costs = simulator_cost_table(train_horizon, "exact", full_simplex_budgets)
full_particle_costs = simulator_cost_table(
    train_horizon,
    "particle",
    full_simplex_budgets,
    flow_particles=particle_flow_particles,
)

full_run_config = pd.DataFrame(
    [
        {
            "device": str(config.device),
            "dtype": str(config.dtype).replace("torch.", ""),
            "n_train": config.n_train,
            "training_runs": config.training_runs,
            "T_train": config.T_train,
            "T_val": config.T_val,
            "hidden_units": config.hidden_units,
            "lr": config.lr,
            "validate_every": config.validate_every,
        }
    ]
)

display(full_run_config)
display(cybersecurity_calibration.round(4))
print("Exact flow, nominal simplex budget")
display(nominal_exact_costs.round(4))
print("Exact flow, full simplex budget")
display(full_exact_costs.round(4))
print("Estimated particle flow, full simplex budget")
display(full_particle_costs.round(4))


,device,dtype,n_train,training_runs,T_train,T_val,hidden_units,lr,validate_every
0,cuda,float64,20000,5,3,50,32,0.001,10


,algorithm,parameter,tv_mean,tv_median,tv_std,tv_p10,tv_p90
0,Simplex,0.1,0.0354,0.0341,0.0153,0.0163,0.0563
1,Simplex,0.2,0.0707,0.0681,0.0306,0.0325,0.1126
2,Simplex,0.4,0.1414,0.1362,0.0612,0.0650,0.2251


Exact flow, nominal simplex budget


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update
algorithm,,,,,,
Simplex,200,10,3,exact,0,630


Exact flow, full simplex budget


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update
algorithm,,,,,,
Simplex,3675,525,3,exact,0,12600


Estimated particle flow, full simplex budget


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update
algorithm,,,,,,
Simplex,3675,525,3,particle,200,13200


## Experiment 1: Exact Flow, Nominal Simplex Budget

Simplex uses `(B, n_aux) = (200, 10)`.


In [10]:
exact_nominal_budget = run_cybersecurity_scenario(
    "simplex_nominal_exact",
    config,
    simplex_lambdas,
    nominal_simplex_budgets,
    cybersecurity_run_plans,
    flow_mode="exact",
    train_horizon=train_horizon,
    validation_horizon=validation_horizon,
    show_progress=True,
    early_stopping_patience=early_stopping_patience,
    max_runtime_seconds=max_runtime_seconds,
)


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update
algorithm,,,,,,
Simplex,200,10,3,exact,0,630


simplex_nominal_exact Simplex lambda=0.1 run=0:   5%|▌         | 1030/20000 [00:52<16:13, 19.48it/s, grad=37.7, value=-0.1629]    


KeyboardInterrupt: 

In [ ]:
exact_nominal_budget_gradient_study = run_cybersecurity_diagnostics(
    exact_nominal_budget,
    config,
    simplex_lambdas,
    cybersecurity_run_plans,
    repetitions=diagnostic_repetitions,
)
display(exact_nominal_budget_gradient_study.round(6))
show_cybersecurity_results(exact_nominal_budget, cybersecurity_calibration)


## Experiment 2: Exact Flow, Full Simplex Budget

Simplex uses `(B, n_aux) = (3675, 525)`, for `12600` simulated transitions per update when `T_train=3`.


In [ ]:
exact_full_budget = run_cybersecurity_scenario(
    "simplex_full_budget_exact",
    config,
    simplex_lambdas,
    full_simplex_budgets,
    cybersecurity_run_plans,
    flow_mode="exact",
    train_horizon=train_horizon,
    validation_horizon=validation_horizon,
    show_progress=True,
    early_stopping_patience=early_stopping_patience,
    max_runtime_seconds=max_runtime_seconds,
)


In [ ]:
exact_full_budget_gradient_study = run_cybersecurity_diagnostics(
    exact_full_budget,
    config,
    simplex_lambdas,
    cybersecurity_run_plans,
    repetitions=diagnostic_repetitions,
)
display(exact_full_budget_gradient_study.round(6))
show_cybersecurity_results(exact_full_budget, cybersecurity_calibration)


## Experiment 3: Estimated Flow, Full Simplex Budget

Training conditions on a particle-estimated population flow with `flow_particles=200`. Validation remains deterministic and exact.


In [ ]:
estimated_full_budget = run_cybersecurity_scenario(
    "simplex_full_budget_estimated_flow",
    config,
    simplex_lambdas,
    full_simplex_budgets,
    cybersecurity_run_plans,
    flow_mode="particle",
    flow_particles=particle_flow_particles,
    train_horizon=train_horizon,
    validation_horizon=validation_horizon,
    show_progress=True,
    early_stopping_patience=early_stopping_patience,
    max_runtime_seconds=max_runtime_seconds,
)


In [ ]:
estimated_full_budget_gradient_study = run_cybersecurity_diagnostics(
    estimated_full_budget,
    config,
    simplex_lambdas,
    cybersecurity_run_plans,
    repetitions=diagnostic_repetitions,
)
display(estimated_full_budget_gradient_study.round(6))
show_cybersecurity_results(estimated_full_budget, cybersecurity_calibration)


## Final Summary

The tables below collect experiment-level summaries and gradient diagnostics. They assume all experiment cells above have been executed.


In [ ]:
cross_protocol_summary = pd.concat(
    [
        scenario_summary_table(exact_nominal_budget, "simplex_nominal_exact"),
        scenario_summary_table(exact_full_budget, "simplex_full_budget_exact"),
        scenario_summary_table(estimated_full_budget, "simplex_full_budget_estimated_flow"),
    ],
    ignore_index=True,
)

all_gradient_diagnostics = pd.concat(
    [
        exact_nominal_budget_gradient_study.assign(experiment="simplex_nominal_exact"),
        exact_full_budget_gradient_study.assign(experiment="simplex_full_budget_exact"),
        estimated_full_budget_gradient_study.assign(experiment="simplex_full_budget_estimated_flow"),
    ],
    ignore_index=True,
)

display(cross_protocol_summary.round(6))
display(all_gradient_diagnostics.round(6))
